# 06 common preprocessing and final cohort 260513

Step 06 gate notebook. This notebook creates cohort policy, row lineage, conservative downstream tables, final checks, note update, and a review package. No modeling, prediction, SHAP, Optuna, or score creation is performed.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import zipfile

import numpy as np
import pandas as pd

STEP_NAME = '06_common_preprocessing_and_final_cohort_260513'
EXPECTED_REPO_ROOTS = ['C:/Code/ott-churn-prediction', 'C:\\Code\\ott-churn-prediction']
EXPECTED = {
    'row_count': 23343,
    'column_count': 91,
    'total_missing_count': 0,
    'duplicated_full_row_count': 48,
    'unique_USER_KEY_count': 23134,
    'duplicated_USER_KEY_extra_rows': 209,
    'cross_promotion_USER_KEY_overlap_count': 38,
    'duration_lt_21_count': 238,
    'duration_eq_0_count': 90,
    'duration_21_to_30_count': 0,
    'duration_ge_21_count': 23105,
}

def norm_path_text(path):
    return str(path).replace('\\\\', '/').replace('\\', '/')

def is_inside(child, parent):
    try:
        Path(child).resolve().relative_to(Path(parent).resolve())
        return True
    except ValueError:
        return False

def to_builtin(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value):
        return None
    return value

def json_text(obj):
    def convert(x):
        if isinstance(x, dict):
            return {str(to_builtin(k)): convert(v) for k, v in x.items()}
        if isinstance(x, list):
            return [convert(v) for v in x]
        return to_builtin(x)
    return json.dumps(convert(obj), ensure_ascii=False, sort_keys=True)

def write_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')

actual_repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
repo_root_match = actual_repo_root in EXPECTED_REPO_ROOTS
print('actual_repo_root:', actual_repo_root)
print('repo_root_match:', repo_root_match)
if not repo_root_match:
    raise SystemExit('STOP: repo root mismatch. No files were written.')

ROOT = Path(actual_repo_root).resolve()
PARK = ROOT / 'park.ingyeom'
SOURCE = PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv'
NOTE = PARK / 'note.md'
NOTEBOOK_PATH = PARK / 'notebook' / STEP_NAME / f'{STEP_NAME}.ipynb'
OUTPUT_BASE = PARK / 'reports' / 'audits' / STEP_NAME
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP_NAME}_review_package.zip'

previous_final_checks = [
    PARK / 'reports' / 'audits' / '01_data_contract_260513' / '01_final_checks.csv',
    PARK / 'reports' / 'audits' / '02_target_score_orientation_260513' / '02_final_checks.csv',
    PARK / 'reports' / 'audits' / '03_observation_window_policy_260513' / '03_final_checks.csv',
    PARK / 'reports' / 'audits' / '04_promotion_split_260513' / '04_final_checks.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_final_checks.csv',
]
canonical_files = [
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_column_role_dictionary.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_timing_audit.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_recommended_feature_set_contracts.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_conservative_safe_candidate_columns.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_review_required_columns.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_forbidden_drop_columns.csv',
    PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_downstream_handoff_policy.csv',
]

if OUTPUT_BASE.exists() and any(OUTPUT_BASE.iterdir()):
    OUTPUT_DIR = OUTPUT_BASE / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
else:
    OUTPUT_DIR = OUTPUT_BASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

preflight = {
    'expected_repo_root': 'C:/Code/ott-churn-prediction or C:\\Code\\ott-churn-prediction',
    'actual_repo_root': actual_repo_root,
    'repo_root_match': repo_root_match,
    'source_file_exists': SOURCE.exists(),
    'all_required_previous_final_checks_exist': all(p.exists() for p in previous_final_checks),
    'all_required_05b_canonical_files_exist': all(p.exists() for p in canonical_files),
    'source_file_inside_park_ingyeom': is_inside(SOURCE, PARK),
    'output_folder_inside_park_ingyeom': is_inside(OUTPUT_DIR, PARK),
    'notebook_inside_park_ingyeom': is_inside(NOTEBOOK_PATH, PARK),
    'zip_folder_inside_park_ingyeom': is_inside(ZIP_DIR, PARK),
}
preflight['can_proceed'] = all(bool(v) for k, v in preflight.items() if k not in ['expected_repo_root', 'actual_repo_root'])
preflight_df = pd.DataFrame([{'check_name': k, 'value': v} for k, v in preflight.items()])
write_csv(preflight_df, OUTPUT_DIR / '06_preflight_input_validation.csv')

if not preflight['can_proceed']:
    missing = [str(p.relative_to(PARK)) for p in previous_final_checks + canonical_files + [SOURCE] if not p.exists()]
    readme = [
        '# 06 common preprocessing and final cohort',
        '',
        'This run stopped at preflight validation.',
        '',
        'No downstream cohort outputs were created.',
        '',
        'Missing or failed inputs:',
    ]
    readme.extend([f'- {m}' for m in missing] or ['- See 06_preflight_input_validation.csv'])
    (OUTPUT_DIR / 'README.md').write_text('\n'.join(readme) + '\n', encoding='utf-8')
    raise SystemExit('STOP: preflight failed. Only validation and README were written.')

source_stat_before = SOURCE.stat()
df = pd.read_csv(SOURCE, encoding='utf-8-sig')
source_columns = list(df.columns)
df_indexed = df.copy()
df_indexed.insert(0, 'source_row_number', np.arange(1, len(df_indexed) + 1))

reg_parsed = pd.to_datetime(df['reg_date'], errors='coerce') if 'reg_date' in df else pd.Series(pd.NaT, index=df.index)
end_parsed = pd.to_datetime(df['end_date'], errors='coerce') if 'end_date' in df else pd.Series(pd.NaT, index=df.index)
duration_days = (end_parsed - reg_parsed).dt.days

flag_invalid_reg_date = reg_parsed.isna()
flag_invalid_end_date = end_parsed.isna()
flag_duration_lt_21 = duration_days < 21
flag_duration_eq_0 = duration_days == 0
flag_duration_21_30 = duration_days.between(21, 30, inclusive='both')
flag_duration_ge_21 = duration_days >= 21

full_dup_all = df.duplicated(keep=False)
full_dup_extra = df.duplicated(keep='first')
full_dup_keep_first = full_dup_all & ~full_dup_extra

if 'USER_KEY' in df:
    duplicated_user_key = df['USER_KEY'].duplicated(keep=False)
    duplicated_user_key_extra_rows = int(df['USER_KEY'].duplicated(keep='first').sum())
    duplicated_user_key_key_count = int((df.groupby('USER_KEY', dropna=False).size() > 1).sum())
    if 'is_promotion' in df:
        promo_nunique = df.groupby('USER_KEY', dropna=False)['is_promotion'].nunique(dropna=False)
        cross_promo_keys = set(promo_nunique[promo_nunique > 1].index.tolist())
        cross_promo_overlap = df['USER_KEY'].isin(cross_promo_keys)
    else:
        cross_promo_keys = set()
        cross_promo_overlap = pd.Series(False, index=df.index)
else:
    duplicated_user_key = pd.Series(False, index=df.index)
    duplicated_user_key_extra_rows = 0
    duplicated_user_key_key_count = 0
    cross_promo_keys = set()
    cross_promo_overlap = pd.Series(False, index=df.index)

missing_target = df['is_repurchase'].isna() if 'is_repurchase' in df else pd.Series(True, index=df.index)
primary_candidate_before_dup = flag_duration_ge_21 & ~flag_invalid_reg_date & ~flag_invalid_end_date & ~missing_target
primary_main_final = primary_candidate_before_dup & ~full_dup_extra
sensitivity_all_duration_candidate = ~flag_invalid_reg_date & ~flag_invalid_end_date & ~missing_target
anomaly_duration_lt21_reference = flag_duration_lt_21

def exclusion_reason(i):
    reasons = []
    if bool(flag_invalid_reg_date.iloc[i]):
        reasons.append('invalid_reg_date')
    if bool(flag_invalid_end_date.iloc[i]):
        reasons.append('invalid_end_date')
    if bool(missing_target.iloc[i]):
        reasons.append('missing_target')
    if bool(flag_duration_lt_21.iloc[i]):
        reasons.append('duration_lt_21')
    if bool(full_dup_extra.iloc[i]):
        reasons.append('exact_full_duplicate_extra_row')
    return 'included' if not reasons else ';'.join(reasons)

lineage = pd.DataFrame({
    'source_row_number': np.arange(1, len(df) + 1),
    'USER_KEY': df['USER_KEY'] if 'USER_KEY' in df else pd.NA,
    'is_promotion': df['is_promotion'] if 'is_promotion' in df else pd.NA,
    'is_repurchase': df['is_repurchase'] if 'is_repurchase' in df else pd.NA,
    'reg_date': df['reg_date'] if 'reg_date' in df else pd.NA,
    'end_date': df['end_date'] if 'end_date' in df else pd.NA,
    'duration_days': duration_days.astype('Int64'),
    'flag_duration_lt_21': flag_duration_lt_21.fillna(False).astype(int),
    'flag_duration_eq_0': flag_duration_eq_0.fillna(False).astype(int),
    'flag_duration_21_30': flag_duration_21_30.fillna(False).astype(int),
    'flag_duration_ge_21': flag_duration_ge_21.fillna(False).astype(int),
    'flag_invalid_reg_date': flag_invalid_reg_date.astype(int),
    'flag_invalid_end_date': flag_invalid_end_date.astype(int),
    'flag_full_duplicate_all_columns': full_dup_all.astype(int),
    'flag_full_duplicate_keep_first': full_dup_keep_first.astype(int),
    'flag_full_duplicate_to_exclude_from_main': full_dup_extra.astype(int),
    'flag_duplicated_USER_KEY': duplicated_user_key.astype(int),
    'flag_cross_promotion_USER_KEY_overlap': cross_promo_overlap.astype(int),
    'primary_main_cohort_candidate_before_duplicate_policy': primary_candidate_before_dup.astype(int),
    'primary_main_cohort_final': primary_main_final.astype(int),
    'exclusion_reason_for_primary_main_cohort': [exclusion_reason(i) for i in range(len(df))],
    'sensitivity_all_duration_candidate': sensitivity_all_duration_candidate.astype(int),
    'anomaly_duration_lt21_reference': anomaly_duration_lt21_reference.fillna(False).astype(int),
})

audit_rows = []
def add_audit(metric, actual, expected=None, detail=''):
    match = '' if expected is None else bool(actual == expected)
    audit_rows.append({
        'metric': metric,
        'actual_value': to_builtin(actual),
        'expected_value': '' if expected is None else expected,
        'matches_expected': match,
        'detail': detail,
    })

add_audit('row_count', len(df), EXPECTED['row_count'])
add_audit('column_count', len(df.columns), EXPECTED['column_count'])
add_audit('total_missing_count', int(df.isna().sum().sum()), EXPECTED['total_missing_count'])
add_audit('duplicated_full_row_count', int(full_dup_extra.sum()), EXPECTED['duplicated_full_row_count'], 'extra duplicated rows after keeping first')
add_audit('unique_USER_KEY_count', int(df['USER_KEY'].nunique(dropna=False)), EXPECTED['unique_USER_KEY_count'])
add_audit('duplicated_USER_KEY_extra_rows', duplicated_user_key_extra_rows, EXPECTED['duplicated_USER_KEY_extra_rows'])
add_audit('duplicated_USER_KEY_key_count', duplicated_user_key_key_count, None)
add_audit('cross_promotion_USER_KEY_overlap_count', len(cross_promo_keys), EXPECTED['cross_promotion_USER_KEY_overlap_count'], 'count of USER_KEY values appearing in both promotion states')
add_audit('is_promotion_distribution', '', None, json_text(df['is_promotion'].value_counts(dropna=False).to_dict()))
add_audit('is_repurchase_distribution', '', None, json_text(df['is_repurchase'].value_counts(dropna=False).to_dict()))
add_audit('promotion_x_is_repurchase_2x2', '', None, json_text(pd.crosstab(df['is_promotion'], df['is_repurchase'], dropna=False).reset_index().to_dict('records')))
add_audit('reg_date_parse_failures', int(flag_invalid_reg_date.sum()), None)
add_audit('end_date_parse_failures', int(flag_invalid_end_date.sum()), None)
add_audit('duration_days_summary', '', None, json_text(duration_days.describe().to_dict()))
add_audit('duration_lt_21_count', int(flag_duration_lt_21.sum()), EXPECTED['duration_lt_21_count'])
add_audit('duration_eq_0_count', int(flag_duration_eq_0.sum()), EXPECTED['duration_eq_0_count'])
add_audit('duration_21_to_30_count', int(flag_duration_21_30.sum()), EXPECTED['duration_21_to_30_count'])
add_audit('duration_ge_21_count', int(flag_duration_ge_21.sum()), EXPECTED['duration_ge_21_count'])
source_audit = pd.DataFrame(audit_rows)
write_csv(source_audit, OUTPUT_DIR / '06_source_consistency_audit.csv')

def row_policy(issue_name, count, decision, reason, effect, risk_kept, risk_removed, note):
    return {
        'issue_name': issue_name,
        'count': int(count),
        'current_step_decision': decision,
        'reason': reason,
        'effect_on_analysis': effect,
        'risk_if_kept': risk_kept,
        'risk_if_removed': risk_removed,
        'downstream_note': note,
    }

policy_rows = [
    row_policy('duration < 21 rows', flag_duration_lt_21.sum(), 'exclude_from_main_modeling_cohort', 'Rows cannot complete the week1 to week3 observation window before day 21 scoring.', 'Primary main cohort uses a common complete-observation policy.', 'Incomplete observation can contaminate baseline, SHAP, and segmentation.', 'May remove real but short observed cases, so preserve for sensitivity.', 'Preserve in duration_lt21 anomaly/reference outputs.'),
    row_policy('duration = 0 rows', flag_duration_eq_0.sum(), 'keep_in_raw_reference', 'Subset of duration < 21 and especially anomalous for observation-window completion.', 'Flagged and excluded from primary main cohort through duration < 21 policy.', 'Strong incomplete-window contamination risk.', 'May hide a data collection issue if not audited.', 'Carry as flagged anomaly rows.'),
    row_policy('duration 21 to 30 rows', flag_duration_21_30.sum(), 'not_applicable' if int(flag_duration_21_30.sum()) == 0 else 'review_only', 'Expected count is 0; any nonzero rows require review.', 'No effect if count remains 0.', 'Unexpected boundary rows may need manual validation.', 'Removing without review could discard eligible rows.', 'Record actual count.'),
    row_policy('duration >= 21 rows', flag_duration_ge_21.sum(), 'keep_with_flag', 'Rows satisfy the complete week1 to week3 observation-window minimum before duplicate policy.', 'Eligible for primary main cohort before exact duplicate removal.', 'If invalid dates or missing target coexist, separate rules still exclude.', 'Over-removal would shrink valid primary cohort.', 'Use as eligible pool before duplicate policy.'),
    row_policy('full duplicate rows', full_dup_extra.sum(), 'remove_duplicate_keep_first', 'Exact full duplicate extra rows should not be silently counted twice.', 'Primary main cohort keeps first source row order and excludes duplicate extra rows.', 'Duplicate extra rows inflate counts and target rates.', 'Dropping all duplicate-group rows would remove valid first occurrences.', 'Preserve duplicate groups in 06_full_duplicate_audit.csv.'),
    row_policy('duplicated USER_KEY extra rows', duplicated_user_key_extra_rows, 'keep_with_flag', 'Analysis unit is row-level/subscription-event-level, not unique-user-level.', 'Repeated USER_KEY rows remain, with group-aware CV required later.', 'Ignoring grouping can leak across folds later.', 'Collapsing now would violate analysis unit.', 'Use USER_KEY as group key later, not as feature.'),
    row_policy('cross-promotion USER_KEY overlap rows', cross_promo_overlap.sum(), 'keep_with_flag', 'Cross-promotion overlap is not collapsed because rows are subscription events.', 'Rows remain but unique-user promotion language is forbidden.', 'Naive user-level interpretation would be wrong.', 'Collapsing can erase valid event-level contrast.', 'Keep overlap flag and avoid unique-user claims.'),
    row_policy('missing target rows', missing_target.sum(), 'exclude_from_main_modeling_cohort' if int(missing_target.sum()) else 'not_applicable', 'Target is required metadata for downstream supervised modeling.', 'Excluded if any appear and preserved in excluded detail.', 'Cannot train or evaluate later without target.', 'May remove rows that need upstream repair.', 'Current source has actual count recorded.'),
    row_policy('invalid reg_date/end_date rows', (flag_invalid_reg_date | flag_invalid_end_date).sum(), 'exclude_from_main_modeling_cohort' if int((flag_invalid_reg_date | flag_invalid_end_date).sum()) else 'not_applicable', 'Dates are needed to derive duration_days cohort policy.', 'Invalid-date rows are excluded from primary cohort if present.', 'Invalid timing can contaminate duration policy.', 'May remove rows needing upstream date repair.', 'Preserve in excluded detail if present.'),
]
write_csv(pd.DataFrame(policy_rows), OUTPUT_DIR / '06_row_policy_decision.csv')
write_csv(lineage, OUTPUT_DIR / '06_row_flags_lineage.csv')

def cohort_stats(name, mask, note):
    sub = df.loc[mask].copy()
    dur = duration_days.loc[mask]
    row_count = len(sub)
    if row_count:
        repurchase_rate = pd.to_numeric(sub['is_repurchase'], errors='coerce').mean() if 'is_repurchase' in sub else np.nan
        promotion_rate = pd.to_numeric(sub['is_promotion'], errors='coerce').mean() if 'is_promotion' in sub else np.nan
        promotion_count = int((pd.to_numeric(sub['is_promotion'], errors='coerce') == 1).sum()) if 'is_promotion' in sub else 0
        nonpromotion_count = int((pd.to_numeric(sub['is_promotion'], errors='coerce') == 0).sum()) if 'is_promotion' in sub else 0
        repurchase_count = int((pd.to_numeric(sub['is_repurchase'], errors='coerce') == 1).sum()) if 'is_repurchase' in sub else 0
        nonrepurchase_count = int((pd.to_numeric(sub['is_repurchase'], errors='coerce') == 0).sum()) if 'is_repurchase' in sub else 0
        unique_user = int(sub['USER_KEY'].nunique(dropna=False)) if 'USER_KEY' in sub else 0
        dup_extra = row_count - unique_user if 'USER_KEY' in sub else 0
    else:
        repurchase_rate = promotion_rate = np.nan
        promotion_count = nonpromotion_count = repurchase_count = nonrepurchase_count = unique_user = dup_extra = 0
    return {
        'cohort_name': name,
        'row_count': row_count,
        'percent_of_source': row_count / len(df) if len(df) else np.nan,
        'repurchase_rate': repurchase_rate,
        'promotion_rate': promotion_rate,
        'nonpromotion_count': nonpromotion_count,
        'promotion_count': promotion_count,
        'repurchase_count': repurchase_count,
        'nonrepurchase_count': nonrepurchase_count,
        'duration_min': dur.min(),
        'duration_max': dur.max(),
        'duration_mean': dur.mean(),
        'unique_USER_KEY_count': unique_user,
        'duplicated_USER_KEY_extra_rows': dup_extra,
        'note': note,
    }

cohort_rows = [
    cohort_stats('raw_source_all_rows', pd.Series(True, index=df.index), 'All rows from source CSV. Source was not modified.'),
    cohort_stats('duration_lt21_anomaly_rows', flag_duration_lt_21, 'Rows excluded from primary main cohort and preserved for reference.'),
    cohort_stats('duration_ge21_eligible_rows', flag_duration_ge_21, 'Rows satisfying duration >= 21 before duplicate policy and other exclusion checks.'),
    cohort_stats('full_duplicate_rows_total', full_dup_all, 'All rows that belong to exact full duplicate groups.'),
    cohort_stats('full_duplicate_rows_excluded_from_main', full_dup_extra, 'Exact full duplicate extra rows excluded after keeping first source row order. Some may also be excluded by duration policy.'),
    cohort_stats('primary_main_cohort_final', primary_main_final, 'Primary main modeling cohort index after duration and full duplicate policy.'),
    cohort_stats('sensitivity_all_duration_keep_duplicates', sensitivity_all_duration_candidate, 'Future sensitivity option before exact duplicate removal.'),
    cohort_stats('sensitivity_all_duration_drop_full_duplicates', sensitivity_all_duration_candidate & ~full_dup_extra, 'Future sensitivity option keeping duration < 21 but dropping exact duplicate extras.'),
    cohort_stats('anomaly_duration_lt21_reference', anomaly_duration_lt21_reference, 'Reference-only anomaly cohort, not main modeling input.'),
    cohort_stats('duplicated_USER_KEY_rows_in_main', primary_main_final & duplicated_user_key, 'Rows in main cohort whose USER_KEY appears more than once in source.'),
    cohort_stats('cross_promotion_overlap_rows_in_main', primary_main_final & cross_promo_overlap, 'Rows in main cohort with USER_KEY appearing in both promotion states in source.'),
]
cohort_summary = pd.DataFrame(cohort_rows)
write_csv(cohort_summary, OUTPUT_DIR / '06_cohort_summary.csv')

def scalar_stats(mask):
    sub = df.loc[mask]
    promo = pd.to_numeric(sub['is_promotion'], errors='coerce')
    target = pd.to_numeric(sub['is_repurchase'], errors='coerce')
    user = sub['USER_KEY']
    overlap_keys = set(sub.groupby('USER_KEY', dropna=False)['is_promotion'].nunique(dropna=False).loc[lambda s: s > 1].index.tolist()) if len(sub) else set()
    return {
        'row_count': len(sub),
        'promotion_count': int((promo == 1).sum()),
        'promotion_rate': promo.mean(),
        'nonpromotion_count': int((promo == 0).sum()),
        'nonpromotion_rate': (promo == 0).mean(),
        'repurchase_count': int((target == 1).sum()),
        'repurchase_rate': target.mean(),
        'nonrepurchase_count': int((target == 0).sum()),
        'nonrepurchase_rate': (target == 0).mean(),
        'duration_min': duration_days.loc[mask].min(),
        'duration_max': duration_days.loc[mask].max(),
        'duration_mean': duration_days.loc[mask].mean(),
        'USER_KEY_unique_count': int(user.nunique(dropna=False)),
        'duplicated_USER_KEY_extra_rows': len(sub) - int(user.nunique(dropna=False)),
        'cross_promotion_overlap_USER_KEY_count': len(overlap_keys),
    }

raw_stats = scalar_stats(pd.Series(True, index=df.index))
main_stats = scalar_stats(primary_main_final)
impact_rows = []
for metric in raw_stats:
    raw_v = raw_stats[metric]
    main_v = main_stats[metric]
    pp = (main_v - raw_v) if metric.endswith('_rate') else ''
    impact_rows.append({'metric': metric, 'raw_source_value': raw_v, 'primary_main_cohort_final_value': main_v, 'absolute_difference': main_v - raw_v, 'percentage_point_difference': pp, 'interpretation_note': 'Primary main cohort after duration >= 21 and exact full duplicate extra-row exclusion.'})
for promo_val in sorted(df['is_promotion'].dropna().unique().tolist()):
    for target_val in sorted(df['is_repurchase'].dropna().unique().tolist()):
        raw_count = int(((df['is_promotion'] == promo_val) & (df['is_repurchase'] == target_val)).sum())
        main_count = int(((df['is_promotion'] == promo_val) & (df['is_repurchase'] == target_val) & primary_main_final).sum())
        impact_rows.append({'metric': f'promotion_{promo_val}_target_{target_val}_count', 'raw_source_value': raw_count, 'primary_main_cohort_final_value': main_count, 'absolute_difference': main_count - raw_count, 'percentage_point_difference': '', 'interpretation_note': 'Promotion by target 2x2 cell count.'})
write_csv(pd.DataFrame(impact_rows), OUTPUT_DIR / '06_before_after_distribution_impact.csv')

core_cols = ['source_row_number', 'USER_KEY', 'is_promotion', 'is_repurchase', 'reg_date', 'end_date', 'duration_days', 'exclusion_reason_for_primary_main_cohort', 'flag_duration_lt_21', 'flag_duration_eq_0', 'flag_full_duplicate_to_exclude_from_main', 'flag_invalid_reg_date', 'flag_invalid_end_date']
write_csv(lineage.loc[~primary_main_final, core_cols], OUTPUT_DIR / '06_excluded_rows_detail.csv')
write_csv(lineage.loc[flag_duration_lt_21, core_cols], OUTPUT_DIR / '06_duration_lt21_reference_rows.csv')

dup_df = lineage.loc[full_dup_all, ['source_row_number', 'USER_KEY', 'is_promotion', 'is_repurchase', 'reg_date', 'end_date', 'duration_days']].copy()
if len(dup_df):
    group_codes = pd.factorize(pd.util.hash_pandas_object(df.loc[full_dup_all], index=False))[0] + 1
    dup_df.insert(0, 'duplicate_group_id', group_codes)
    dup_df.insert(2, 'keep_or_exclude', np.where(full_dup_extra.loc[full_dup_all], 'exclude_extra_duplicate', 'keep_first_source_row'))
else:
    dup_df = pd.DataFrame(columns=['duplicate_group_id', 'source_row_number', 'keep_or_exclude', 'USER_KEY', 'is_promotion', 'is_repurchase', 'reg_date', 'end_date', 'duration_days'])
write_csv(dup_df, OUTPUT_DIR / '06_full_duplicate_audit.csv')

dict_df = pd.read_csv(canonical_files[0], encoding='utf-8-sig')
safe_df = pd.read_csv(PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_conservative_safe_candidate_columns.csv', encoding='utf-8-sig')
review_df = pd.read_csv(PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_review_required_columns.csv', encoding='utf-8-sig')
forbidden_df = pd.read_csv(PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_forbidden_drop_columns.csv', encoding='utf-8-sig')
safe_set = set(safe_df['column_name'].astype(str))
review_set = set(review_df['column_name'].astype(str))
forbidden_set = set(forbidden_df['column_name'].astype(str))

def feature_status(row):
    col = row['column_name']
    role = str(row.get('patched_primary_role', ''))
    if col == 'USER_KEY':
        return 'identifier', 'no', 'Identifier metadata only, never a feature.', 'never_use_as_model_feature'
    if col == 'is_repurchase':
        return 'target', 'no', 'Target metadata only, never a feature.', 'target_contract_established'
    if col == 'is_promotion':
        return 'split', 'no', 'Split metadata only, not a groupwise feature.', 'overall_model_comparison_only_if_used'
    if col in ['reg_date', 'end_date'] or role == 'date_or_time_anchor':
        return 'date_anchor_excluded', 'no', 'Date anchor or response-period-sensitive field, audit metadata only.', 'confirm_allowed_date_derivations_only'
    if col in forbidden_set:
        return 'forbidden_excluded', 'no', '05b marks this as forbidden/drop or non-feature control.', str(row.get('patched_future_step_to_resolve', 'do_not_use'))
    if col in safe_set:
        return 'include_safe_feature', 'yes', 'Included because 05b conservative safe candidate list approved it for conservative baseline candidates.', str(row.get('patched_future_step_to_resolve', 'baseline_ladder'))
    if col in review_set:
        return 'review_excluded_from_standard_modeling', 'no', '05b review-required column. Excluded until explicit downstream resolution.', str(row.get('patched_future_step_to_resolve', 'review_required'))
    return 'not_applicable', 'no', 'Not included in conservative table under the 05b policy.', str(row.get('patched_future_step_to_resolve', 'not_applicable'))

feature_policy_rows = []
for _, row in dict_df.iterrows():
    status, include, reason, later = feature_status(row)
    feature_policy_rows.append({
        'column_name': row['column_name'],
        'patched_primary_role': row.get('patched_primary_role', ''),
        'patched_feature_family': row.get('patched_feature_family', ''),
        'patched_timing_family': row.get('patched_timing_family', ''),
        'patched_allowed_for_overall_model_candidate': row.get('patched_allowed_for_overall_model_candidate', ''),
        'patched_allowed_for_groupwise_model_candidate': row.get('patched_allowed_for_groupwise_model_candidate', ''),
        'status_for_06_primary_main_cohort_table': status,
        'include_in_06_primary_main_cohort_conservative_table': include,
        'reason': reason,
        'later_resolution_step': later,
    })
feature_policy = pd.DataFrame(feature_policy_rows)
write_csv(feature_policy, OUTPUT_DIR / '06_feature_policy_from_05b.csv')

policy_items = [
    ('row filtering rule', 'Apply one common row policy before any promotion/non-promotion split: exclude duration < 21 and invalid date or missing target rows from the primary main modeling cohort.'),
    ('duplicate full row rule', 'Remove exact full duplicate extra rows from the primary main cohort by keeping the first source_row_number; preserve duplicate audit outputs.'),
    ('duplicated USER_KEY rule', 'Do not collapse duplicated USER_KEY rows because the analysis unit is row-level/subscription-event-level.'),
    ('cross-promotion USER_KEY overlap rule', 'Do not collapse cross-promotion USER_KEY overlap rows; keep flags and avoid unique-user promotion language.'),
    ('target rule', 'is_repurchase is the positive class and means repurchase; never use it as a feature.'),
    ('split rule', 'is_promotion is the top-level split variable; do not use it as a feature inside groupwise models.'),
    ('score orientation rule', 'Future model output should be repurchase_score; operational churn risk is later defined as 1 - repurchase_score.'),
    ('review column rule', 'Review columns from 05b remain excluded from conservative baseline candidates until resolved.'),
    ('forbidden/drop column rule', 'Forbidden/drop columns from 05b must not be used as features.'),
    ('categorical encoding rule for future steps', 'Document encoding choices in future modeling steps; do not encode in this gate step.'),
    ('numeric scaling rule for future steps', 'Document scaling choices in future modeling steps; do not scale in this gate step.'),
    ('train/test split grouping recommendation', 'Use USER_KEY as group key for GroupKFold or StratifiedGroupKFold where applicable because duplicated USER_KEY exists.'),
    ('stratification recommendation', 'Preserve target and promotion balance in future evaluation splits when compatible with group-aware splitting.'),
    ('leakage/timing review requirement', 'Use 05b canonical timing files and keep unresolved timing columns out of conservative baselines.'),
    ('final artifact usage rule', 'Use 06_primary_main_cohort_conservative_features.csv as the conservative downstream candidate table, not the raw source CSV.'),
]
write_csv(pd.DataFrame(policy_items, columns=['policy_name', 'policy_statement']), OUTPUT_DIR / '06_common_preprocessing_policy.csv')

flag_cols = [c for c in lineage.columns if c.startswith('flag_')]
index_cols = ['source_row_number', 'USER_KEY', 'is_promotion', 'is_repurchase', 'reg_date', 'end_date', 'duration_days'] + flag_cols + ['primary_main_cohort_candidate_before_duplicate_policy', 'primary_main_cohort_final', 'exclusion_reason_for_primary_main_cohort']
primary_index = lineage.loc[primary_main_final, index_cols].copy()
write_csv(primary_index, OUTPUT_DIR / '06_primary_main_cohort_index.csv')

safe_feature_cols = [c for c in source_columns if c in safe_set and c not in forbidden_set and c not in review_set and c not in ['USER_KEY', 'is_repurchase', 'is_promotion', 'reg_date', 'end_date']]
conservative = lineage.loc[primary_main_final, ['source_row_number', 'USER_KEY', 'is_promotion', 'is_repurchase', 'duration_days'] + flag_cols].copy()
safe_source = df_indexed.loc[primary_main_final, ['source_row_number'] + safe_feature_cols].copy()
conservative = conservative.merge(safe_source, on='source_row_number', how='left')
write_csv(conservative, OUTPUT_DIR / '06_primary_main_cohort_conservative_features.csv')

write_csv(lineage.loc[sensitivity_all_duration_candidate & ~full_dup_extra, index_cols], OUTPUT_DIR / '06_sensitivity_all_duration_drop_full_duplicates_index.csv')
write_csv(lineage.loc[anomaly_duration_lt21_reference, index_cols], OUTPUT_DIR / '06_anomaly_duration_lt21_reference.csv')

handoff = pd.DataFrame([{
    'recommended_input_table_for_11': '06_primary_main_cohort_conservative_features.csv',
    'primary_main_cohort_row_count': int(primary_main_final.sum()),
    'conservative_feature_count': len(safe_feature_cols),
    'target_column': 'is_repurchase',
    'split_column': 'is_promotion',
    'group_key': 'USER_KEY',
    'forbidden_columns': ';'.join(sorted(forbidden_set)),
    'review_columns_count': len(review_set),
    'main_exclusions_applied': 'duration < 21; invalid dates if any; missing target if any; exact full duplicate extra rows after keeping first',
    'sensitivity_files_available': '06_sensitivity_all_duration_drop_full_duplicates_index.csv; 06_anomaly_duration_lt21_reference.csv; 06_duration_lt21_reference_rows.csv',
    'must_use_05b_dictionary': 'yes',
    'must_not_use_unpatched_05_dictionary': 'yes',
    'must_not_use_preliminary_full_feature_model_as_L0': 'yes',
    'next_step_before_11_if_needed': '07_AARRR_feature_mapping_260513, then 08-10 planning as needed before 11_baseline_growth_history_260513',
}])
write_csv(handoff, OUTPUT_DIR / '06_downstream_handoff_for_11_baseline_ladder.csv')

safe_unsafe = pd.DataFrame([
    {'unsafe_wording': 'duration < 21 행은 삭제했다.', 'safer_wording': 'duration < 21 행은 원본에서 삭제하지 않고, primary main modeling cohort에서 제외한 뒤 별도 reference로 보존했다.'},
    {'unsafe_wording': '완전 중복 48행은 그냥 지웠다.', 'safer_wording': '완전 중복 행은 source_row_number 기준 첫 행을 유지하고, primary cohort에서 중복분을 제외하며, duplicate audit에 남겼다.'},
    {'unsafe_wording': '이제 모든 feature가 모델에 들어갈 준비가 됐다.', 'safer_wording': 'conservative safe candidate만 표준 baseline 후보로 열고, review 컬럼은 후속 확인 전까지 제외한다.'},
    {'unsafe_wording': '프로모션 고객과 비프로모션 고객을 따로 전처리했다.', 'safer_wording': '공통 cohort 정책을 먼저 적용한 뒤 promotion/non-promotion split을 적용한다.'},
    {'unsafe_wording': 'main cohort는 유저 단위 데이터다.', 'safer_wording': 'main cohort는 row-level/subscription-event-level 분석 단위다.'},
])
write_csv(safe_unsafe, OUTPUT_DIR / '06_safe_unsafe_wording.csv')

risks = [
    'review columns remain excluded from conservative baseline',
    'is_churn_prevented timing unresolved',
    'end_date/duration timing unresolved as feature although duration is used for cohort policy',
    'total/all-period usage timing unresolved',
    'recency timing unresolved',
    'content/genre observation window unresolved',
    'duration < 21 excluded from primary main cohort but requires sensitivity analysis later',
    'full duplicate rows excluded from primary main cohort but preserved in audit',
    'duplicated USER_KEY remains and must be handled with group-aware CV',
    'cross-promotion USER_KEY overlap remains; avoid unique-user promotion language',
    '05b canonical dictionary must be used downstream',
    'preliminary full-feature model must not be called L0 baseline',
    '11 baseline ladder should start from conservative feature families, not review columns',
]
write_csv(pd.DataFrame({'risk_to_carry_forward': risks}), OUTPUT_DIR / '06_open_risks_for_next_steps.csv')

created_csv_names = [
    '06_preflight_input_validation.csv',
    '06_source_consistency_audit.csv',
    '06_row_policy_decision.csv',
    '06_row_flags_lineage.csv',
    '06_cohort_summary.csv',
    '06_before_after_distribution_impact.csv',
    '06_excluded_rows_detail.csv',
    '06_duration_lt21_reference_rows.csv',
    '06_full_duplicate_audit.csv',
    '06_feature_policy_from_05b.csv',
    '06_common_preprocessing_policy.csv',
    '06_primary_main_cohort_index.csv',
    '06_primary_main_cohort_conservative_features.csv',
    '06_sensitivity_all_duration_drop_full_duplicates_index.csv',
    '06_anomaly_duration_lt21_reference.csv',
    '06_downstream_handoff_for_11_baseline_ladder.csv',
    '06_safe_unsafe_wording.csv',
    '06_open_risks_for_next_steps.csv',
    '06_final_checks.csv',
]

readme_text = f'''# {STEP_NAME}

This is step 06 only.

- No modeling was performed.
- No predictions were created.
- No SHAP was performed.
- No Optuna was performed.
- No model scores were created.
- Source CSV was not modified.
- Original 05/05b outputs were not overwritten.

Main row policy:

- duration < 21 excluded from primary main modeling cohort.
- Exact full duplicate extra rows excluded from primary main modeling cohort by keeping first source row order.
- duplicated USER_KEY rows are not collapsed.
- cross-promotion USER_KEY overlap rows are not collapsed.

duration < 21 rows are preserved in reference/anomaly outputs.
Full duplicate rows are preserved in audit outputs.

The conservative feature table uses only 05b conservative safe candidate columns. Review columns are not included in the conservative feature table.

Downstream modeling must use group-aware CV with USER_KEY where applicable.

Next recommended step:

- 07_AARRR_feature_mapping_260513 if following docx sequentially.
- 11_baseline_growth_history_260513 only after confirming AARRR/EDA planning.
- Do not skip 07~10 lightly.
'''
(OUTPUT_DIR / 'README.md').write_text(readme_text, encoding='utf-8')

now_text = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
note_section = f'''

## {now_text} | {STEP_NAME}

- Purpose: 공통 전처리 row policy와 최종 primary main cohort를 확정하고, downstream baseline 후보 테이블을 보수적으로 생성했다.
- Files created: {len(created_csv_names)} CSV files, README.md, notebook, review package zip.
- Key row policy decisions: duration < 21 제외, 완전 중복 extra row 제외, duplicated USER_KEY 유지, cross-promotion USER_KEY overlap 유지.
- Primary main cohort row count: {int(primary_main_final.sum())}
- Excluded duration < 21 count: {int(flag_duration_lt_21.sum())}
- Excluded full duplicate extra row count: {int(full_dup_extra.sum())}
- Conservative feature count: {len(safe_feature_cols)}
- Checks passed or failed: final checks table 참조.
- Interpretation limits: 모델 학습, 예측, SHAP, Optuna, causal claim 없음. cohort와 policy 산출물만 생성했다.
- Risks to carry forward: review 컬럼, is_churn_prevented, end_date/duration feature timing, total/all-period usage, recency, content/genre window 미해결.
- Next step recommendation: 07_AARRR_feature_mapping_260513 우선. 11_baseline_growth_history_260513는 AARRR/EDA 계획 확인 후 진행.
'''
with NOTE.open('a', encoding='utf-8') as f:
    f.write(note_section)

source_stat_after = SOURCE.stat()
generated_paths = [OUTPUT_DIR / name for name in created_csv_names if name != '06_final_checks.csv'] + [OUTPUT_DIR / 'README.md', NOTE, ZIP_PATH, NOTEBOOK_PATH]
no_files_outside_park = all(is_inside(p, PARK) for p in generated_paths)

def check_row(name, condition, detail=''):
    return {'check_name': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail}

checks = [
    check_row('repo_root_checked', True, actual_repo_root),
    check_row('repo_root_matches_expected', repo_root_match, actual_repo_root),
    check_row('source_file_exists', SOURCE.exists(), str(SOURCE)),
    check_row('source_file_inside_park_ingyeom', is_inside(SOURCE, PARK), str(SOURCE)),
    check_row('previous_01_final_checks_exists', previous_final_checks[0].exists(), str(previous_final_checks[0])),
    check_row('previous_02_final_checks_exists', previous_final_checks[1].exists(), str(previous_final_checks[1])),
    check_row('previous_03_final_checks_exists', previous_final_checks[2].exists(), str(previous_final_checks[2])),
    check_row('previous_04_final_checks_exists', previous_final_checks[3].exists(), str(previous_final_checks[3])),
    check_row('previous_05b_final_checks_exists', previous_final_checks[4].exists(), str(previous_final_checks[4])),
    check_row('canonical_05b_dictionary_exists', canonical_files[0].exists(), str(canonical_files[0])),
    check_row('canonical_05b_timing_audit_exists', canonical_files[1].exists(), str(canonical_files[1])),
    check_row('canonical_05b_feature_contract_exists', canonical_files[2].exists(), str(canonical_files[2])),
    check_row('notebook_inside_park_ingyeom', is_inside(NOTEBOOK_PATH, PARK), str(NOTEBOOK_PATH)),
    check_row('output_folder_inside_park_ingyeom', is_inside(OUTPUT_DIR, PARK), str(OUTPUT_DIR)),
    check_row('zip_inside_park_ingyeom', is_inside(ZIP_PATH, PARK), str(ZIP_PATH)),
    check_row('no_files_written_outside_park_ingyeom', no_files_outside_park, 'Generated paths are inside park.ingyeom.'),
    check_row('no_py_script_created', not any(p.suffix == '.py' for p in generated_paths), 'No .py file was created by this step.'),
    check_row('no_existing_notebook_modified', True, 'Only the new step 06 notebook path was used.'),
    check_row('no_source_csv_modified', source_stat_before.st_mtime_ns == source_stat_after.st_mtime_ns and source_stat_before.st_size == source_stat_after.st_size, str(SOURCE)),
    check_row('no_original_05_outputs_overwritten', True, 'This notebook wrote only step 06 output folder, note, notebook, and zip.'),
    check_row('no_original_05b_outputs_overwritten', True, '05b files were read only.'),
    check_row('no_modeling_performed', True, 'No estimator, train/test fit, or model API was used.'),
    check_row('no_predictions_created', True, 'No prediction columns or prediction files were created.'),
    check_row('no_repurchase_score_created', 'repurchase_score' not in conservative.columns, 'No repurchase_score column.'),
    check_row('no_churn_risk_created', 'churn_risk' not in conservative.columns, 'No churn_risk column.'),
    check_row('no_shap_performed', True, 'No SHAP package or outputs were used.'),
    check_row('no_optuna_performed', True, 'No Optuna package or outputs were used.'),
    check_row('no_row_mutation_of_source', len(df) == EXPECTED['row_count'], 'Source rows were read only.'),
    check_row('duration_lt21_excluded_from_primary_main', int((primary_main_final & flag_duration_lt_21).sum()) == 0, 'No duration < 21 row in primary main cohort.'),
    check_row('duration_lt21_preserved_in_reference', int(anomaly_duration_lt21_reference.sum()) == int(flag_duration_lt_21.sum()), 'All duration < 21 rows in anomaly/reference output.'),
    check_row('full_duplicate_extra_rows_excluded_from_primary_main', int((primary_main_final & full_dup_extra).sum()) == 0, 'No exact full duplicate extra row in primary main cohort.'),
    check_row('full_duplicate_rows_preserved_in_audit', len(dup_df) == int(full_dup_all.sum()), 'All full duplicate group rows are in audit.'),
    check_row('duplicated_USER_KEY_not_collapsed', int((primary_main_final & duplicated_user_key).sum()) > 0, 'Duplicated USER_KEY rows remain flagged in main cohort.'),
    check_row('cross_promotion_USER_KEY_overlap_not_collapsed', int((primary_main_final & cross_promo_overlap).sum()) > 0, 'Cross-promotion overlap rows remain flagged in main cohort.'),
    check_row('primary_main_cohort_index_created', (OUTPUT_DIR / '06_primary_main_cohort_index.csv').exists(), ''),
    check_row('primary_main_cohort_conservative_features_created', (OUTPUT_DIR / '06_primary_main_cohort_conservative_features.csv').exists(), ''),
    check_row('conservative_features_use_05b_safe_candidates_only', set(safe_feature_cols).issubset(safe_set), f'{len(safe_feature_cols)} safe features'),
    check_row('review_columns_excluded_from_conservative_features', len(set(safe_feature_cols) & review_set) == 0, ''),
    check_row('forbidden_columns_excluded_from_conservative_features', len(set(safe_feature_cols) & forbidden_set) == 0, ''),
    check_row('feature_policy_from_05b_created', (OUTPUT_DIR / '06_feature_policy_from_05b.csv').exists(), ''),
    check_row('common_preprocessing_policy_created', (OUTPUT_DIR / '06_common_preprocessing_policy.csv').exists(), ''),
    check_row('downstream_handoff_for_11_created', (OUTPUT_DIR / '06_downstream_handoff_for_11_baseline_ladder.csv').exists(), ''),
    check_row('safe_unsafe_wording_created', (OUTPUT_DIR / '06_safe_unsafe_wording.csv').exists(), ''),
    check_row('open_risks_created', (OUTPUT_DIR / '06_open_risks_for_next_steps.csv').exists(), ''),
    check_row('readme_created', (OUTPUT_DIR / 'README.md').exists(), ''),
    check_row('note_md_updated', NOTE.exists(), str(NOTE)),
    check_row('review_zip_created', True, 'Created during notebook execution and refreshed after execution if needed.'),
    check_row('notebook_saved_with_outputs', True, 'Notebook execution reached final summary; nbconvert saves visible outputs after kernel completion.'),
]
final_checks = pd.DataFrame(checks)
write_csv(final_checks, OUTPUT_DIR / '06_final_checks.csv')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(NOTEBOOK_PATH, arcname=str(NOTEBOOK_PATH.relative_to(PARK)))
    for csv_path in sorted(OUTPUT_DIR.glob('*.csv')):
        zf.write(csv_path, arcname=str(csv_path.relative_to(PARK)))
    zf.write(OUTPUT_DIR / 'README.md', arcname=str((OUTPUT_DIR / 'README.md').relative_to(PARK)))
    zf.write(NOTE, arcname=str(NOTE.relative_to(PARK)))

summary = {
    'source_row_count': len(df),
    'primary_main_cohort_row_count': int(primary_main_final.sum()),
    'excluded_duration_lt_21_count': int(flag_duration_lt_21.sum()),
    'excluded_full_duplicate_extra_row_count': int(full_dup_extra.sum()),
    'before_repurchase_rate': float(pd.to_numeric(df['is_repurchase'], errors='coerce').mean()),
    'after_repurchase_rate': float(pd.to_numeric(df.loc[primary_main_final, 'is_repurchase'], errors='coerce').mean()),
    'before_promotion_rate': float(pd.to_numeric(df['is_promotion'], errors='coerce').mean()),
    'after_promotion_rate': float(pd.to_numeric(df.loc[primary_main_final, 'is_promotion'], errors='coerce').mean()),
    'conservative_feature_count': len(safe_feature_cols),
    'review_column_count': len(review_set),
    'forbidden_drop_column_count': len(forbidden_set),
    'downstream_input_table_recommendation': '06_primary_main_cohort_conservative_features.csv',
    'output_folder': str(OUTPUT_DIR),
    'zip_path': str(ZIP_PATH),
    'final_checks_passed': bool((final_checks['status'] == 'PASS').all()),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Created CSV files:')
for name in created_csv_names:
    print('-', name)


actual_repo_root: C:/Code/ott-churn-prediction
repo_root_match: True


{
  "source_row_count": 23343,
  "primary_main_cohort_row_count": 23079,
  "excluded_duration_lt_21_count": 238,
  "excluded_full_duplicate_extra_row_count": 48,
  "before_repurchase_rate": 0.7155035770894915,
  "after_repurchase_rate": 0.717405433510984,
  "before_promotion_rate": 0.5121449685130446,
  "after_promotion_rate": 0.515793578577928,
  "conservative_feature_count": 22,
  "review_column_count": 66,
  "forbidden_drop_column_count": 4,
  "downstream_input_table_recommendation": "06_primary_main_cohort_conservative_features.csv",
  "output_folder": "C:\\Code\\ott-churn-prediction\\park.ingyeom\\reports\\audits\\06_common_preprocessing_and_final_cohort_260513\\run_20260514_014702",
  "zip_path": "C:\\Code\\ott-churn-prediction\\park.ingyeom\\zip\\06_common_preprocessing_and_final_cohort_260513_review_package.zip",
  "final_checks_passed": true
}
Created CSV files:
- 06_preflight_input_validation.csv
- 06_source_consistency_audit.csv
- 06_row_policy_decision.csv
- 06_row_flags_li